<a href="https://colab.research.google.com/github/ibtihalalf/Sdaia-Bootcamp/blob/main/C4/M2/Ex2_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information extraction

Docling provides the capability of extracting information, i.e. structured data, from unstructured documents.

The user can provide the desired data schema AKA *template*, either as a dictionary or as a Pydantic model, and Docling will return
the extracted data as a standardized output, organized by page.

Check out the subsections below for different usage scenarios.

In [1]:
%pip install -q docling[vlm]  # Install the Docling package with VLM support

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 73.8 MB/s eta 0:00:00
   ━━

In [2]:
from IPython import display
from pydantic import BaseModel, Field
from rich import print

In this notebook, we will work with an example input image — let's quickly inspect it:

In [7]:
file_path = (
    "https://upload.wikimedia.org/wikipedia/commons/9/9f/Swiss_QR-Bill_example.jpg"
)
display.HTML(f"<img src='{file_path}' height='1000'>")

## Defining the extractor

Let's first define our extractor:

In [8]:
from docling.datamodel.base_models import InputFormat
from docling.document_extractor import DocumentExtractor

extractor = DocumentExtractor(allowed_formats=[InputFormat.IMAGE, InputFormat.PDF])

Following, we look at different ways to define the data template.

## Using a string template

In [ ]:
result = extractor.extract(
    source=file_path,
    template='{"bill_no": "string", "total": "float"}',
)
print(result.pages)

Only PDF and image formats are supported.
  return next(all_res)
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[
    ExtractedPageData(
        page_no=1,
        extracted_data={'bill_no': '3139', 'total': 3949.75},
        raw_text='{"bill_no": "3139", "total": 3949.75}',
        errors=[]
    )
]

## Using a dict template

In [ ]:
result = extractor.extract(
    source=file_path,
    template={
        "bill_no": "string",
        "total": "float",
    },
)
print(result.pages)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[
    ExtractedPageData(
        page_no=1,
        extracted_data={'bill_no': '3139', 'total': 3949.75},
        raw_text='{"bill_no": "3139", "total": 3949.75}',
        errors=[]
    )
]

## Using a Pydantic model template

First we define the Pydantic model we want to use

In [ ]:
from typing import Optional


class Invoice(BaseModel):
    bill_no: str = Field(
        examples=["A123", "5414"]
    )  # provide some examples, but no default value
    total: float = Field(
        default=10, examples=[20]
    )  # provide some examples and a default value
    tax_id: Optional[str] = Field(default=None, examples=["1234567890"])

The class itself can then be used directly as the template:

In [ ]:
result = extractor.extract(
    source=file_path,
    template=Invoice,
)
print(result.pages)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[
    ExtractedPageData(
        page_no=1,
        extracted_data={'bill_no': '3139', 'total': 3949.75, 'tax_id': None},
        raw_text='{"bill_no": "3139", "total": 3949.75, "tax_id": null}',
        errors=[]
    )
]

Alternatively, a Pydantic model instance can be passed as a template instead, allowing to override the default values.

This can be very useful in scenarios where we happen to have available context that is more relevant than the
default values predefined in the model definition.

E.g. in the example below:
- `bill_no` and `total` are actually set from the value extracted from the data,
- there was no `tax_id` to be extracted, so the updated default we provided was applied

In [ ]:
result = extractor.extract(
    source=file_path,
    template=Invoice(
        bill_no="41",
        total=100,
        tax_id="42",
    ),
)
print(result.pages)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[
    ExtractedPageData(
        page_no=1,
        extracted_data={'bill_no': '3139', 'total': 3949.75, 'tax_id': '42'},
        raw_text='{"bill_no": "3139", "total": 3949.75, "tax_id": "42"}',
        errors=[]
    )
]

### Advanced Pydantic model

Besides a flat template, we can in principle use any Pydantic model, thus leveraging reuse and being able to capture
hierarchies:

In [ ]:
class Contact(BaseModel):
    name: Optional[str] = Field(default=None, examples=["Smith"])
    address: str = Field(default="123 Main St", examples=["456 Elm St"])
    postal_code: str = Field(default="12345", examples=["67890"])
    city: str = Field(default="Anytown", examples=["Othertown"])
    country: Optional[str] = Field(default=None, examples=["Canada"])


class ExtendedInvoice(BaseModel):
    bill_no: str = Field(
        examples=["A123", "5414"]
    )  # provide some examples, but not the actual value of the test sample
    total: float = Field(
        default=10, examples=[20]
    )  # provide a default value and some examples
    garden_work_hours: int = Field(default=1, examples=[2])
    sender: Contact = Field(default=Contact(), examples=[Contact()])
    receiver: Contact = Field(default=Contact(), examples=[Contact()])

In [ ]:
result = extractor.extract(
    source=file_path,
    template=ExtendedInvoice,
)
print(result.pages)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[
    ExtractedPageData(
        page_no=1,
        extracted_data={
            'bill_no': '3139',
            'total': 3949.75,
            'garden_work_hours': 28,
            'sender': {
                'name': 'Robert Schneider',
                'address': 'Rue du Lac 1268',
                'postal_code': '2501',
                'city': 'Biel',
                'country': 'Switzerland'
            },
            'receiver': {
                'name': 'Pia Rutschmann',
                'address': 'Marktgasse 28',
                'postal_code': '9400',
                'city': 'Rorschach',
                'country': 'Switzerland'
            }
        },
        raw_text='{"bill_no": "3139", "total": 3949.75, "garden_work_hours": 28, "sender": {"name": "Robert 
Schneider", "address": "Rue du Lac 1268", "postal_code": "2501", "city": "Biel", "country": "Switzerland"}, 
"receiver": {"name": "Pia Rutschmann", "address": "Marktgasse 28", "postal_code": "9400", "city": "Rorschach", 
"country": "Switzerland"}}',
        errors=[]
    )
]

### Validating and loading the extracted data

The generated response data can be easily validated and loaded via Pydantic:

In [ ]:
invoice = ExtendedInvoice.model_validate(result.pages[0].extracted_data)
print(invoice)

ExtendedInvoice(
    bill_no='3139',
    total=3949.75,
    garden_work_hours=28,
    sender=Contact(
        name='Robert Schneider',
        address='Rue du Lac 1268',
        postal_code='2501',
        city='Biel',
        country='Switzerland'
    ),
    receiver=Contact(
        name='Pia Rutschmann',
        address='Marktgasse 28',
        postal_code='9400',
        city='Rorschach',
        country='Switzerland'
    )
)

This way, we can get from completely unstructured data to a very structured and developer-friendly representation:

In [ ]:
print(
    f"Invoice #{invoice.bill_no} was sent by {invoice.sender.name} "
    f"to {invoice.receiver.name} at {invoice.sender.address}."
)

Invoice #3139 was sent by Robert Schneider to Pia Rutschmann at Rue du Lac 1268.

**Task 1**: Information Extraction with a VLM

In [ ]:
%pip install -q docling[vlm]

from typing import List, Optional
from pydantic import BaseModel, Field
from rich import print

from docling.datamodel.base_models import InputFormat
from docling.document_extractor import DocumentExtractor


class Education(BaseModel):
    institution: Optional[str] = Field(default=None)
    degree: Optional[str] = Field(default=None)
    field_of_study: Optional[str] = Field(default=None)
    start_date: Optional[str] = Field(default=None)
    end_date: Optional[str] = Field(default=None)


class WorkExperience(BaseModel):
    company: Optional[str] = Field(default=None)
    job_title: Optional[str] = Field(default=None)
    start_date: Optional[str] = Field(default=None)
    end_date: Optional[str] = Field(default=None)
    responsibilities: List[str] = Field(default_factory=list)


class Resume(BaseModel):
    full_name: Optional[str] = Field(default=None)
    email: Optional[str] = Field(default=None)
    phone: Optional[str] = Field(default=None)
    location: Optional[str] = Field(default=None)
    linkedin: Optional[str] = Field(default=None)
    github: Optional[str] = Field(default=None)

    summary: Optional[str] = Field(default=None)
    education: List[Education] = Field(default_factory=list)
    work_experience: List[WorkExperience] = Field(default_factory=list)
    skills: List[str] = Field(default_factory=list)
    projects: List[str] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)


extractor = DocumentExtractor(
    allowed_formats=[InputFormat.PDF, InputFormat.IMAGE]
)

file_path = "/content/resume.pdf"

result = extractor.extract(
    source=file_path,
    template=Resume,
)

print(result.pages)

**Task 2**: Arabic Document Processing

In [3]:
!pip install transformers qwen_vl_utils accelerate>=0.26.0 PEFT -U
!pip install -U bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.8 MB/s eta 0:00:00


In [11]:
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
import os
from qwen_vl_utils import process_vision_info
# Removed requests and BytesIO since we are loading from local file


model_name = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                torch_dtype="auto",
                device_map="auto"
            )
processor = AutoProcessor.from_pretrained(model_name)
max_tokens = 2000

prompt = "Below is the image of one page of a document, as well as some raw textual content that was previously extracted for it. Just return the plain text representation of this document as if you were reading it naturally. Do not hallucinate."


ts at this path or update the path accordingly.
src = "/content/reciept.png" # Define the local path to the image file
image = Image.open(src) # Open the image directly from the local file

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{src}"},
            {"type": "text", "text": prompt},
        ],
    }
]
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

#  Check for CUDA availability and use CPU if not available ---
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = inputs.to(device)

generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
os.remove(src)
print(output_text)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

<h2><u>إيصال</u> <b>مؤقت</b></h2><br><p>رقم الإيمال: 12345 <b>:</b> <i>النسبة:</i> 1 <i>أغسطس</i> 2024 
<i>الوقت:</i> 19:30</p><br><p>تفاصيل <i>الطلب</i></p><br><h3>الوصف</h3><br><p>العمالي <b>الجمالي</b> <b>السمع</b> 
<i>الكمية</i> <i>السعر</i></p><br><p>30.00 يورو <i>2</i></p><br><p>10.00 يورو <i>1</i></p><br><p>12.00 يورو 
<i>1</i></p><br><p>60.00 يورو <i>3</i></p><br><p>16.00 يورو <i>2</i></p><br><p>128.00 يورو</p><br><h4>شكراً لتناول 
الطعام <i>معنا!</i></h4>